# Batch monitoring: report, metric, notification

This optional example demonstrates one comparison and its handoff to existing services.
The default frames are synthetic. Replace them with your reference/current batch outputs;
your model analysis still owns labels and quality metrics. No model is trained here.

Install `requirements-monitoring.txt` in this notebook environment. Also install
`requirements-tracking.txt` if you enable MLflow below.

- Locate the application source and import the report helpers.
- Use the notebook's normal AWS credentials only if you enable publication later.
- Keep the report dependency separate from the approved training image.


In [ ]:
from pathlib import Path
import json
import sys
import os
import pandas as pd
from IPython.display import FileLink, display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "mlops-models/src").is_dir())
sys.path.insert(0, str(ROOT / "mlops-models/src"))
from mlops_models.monitoring.report import write_report, publish_report


## Choose the batches and feature columns

- Create 500 synthetic rows and a second batch with a deliberate activity shift.
- Keep the segment distribution unchanged so the report distinguishes the affected column.
- Supply quality metrics from your prediction analysis when available; `None` records no quality claim.
- Record the actual model, batch and S3 references when replacing the sample data.


In [ ]:
reference = pd.DataFrame({"activity": [float(i % 50) for i in range(500)], "segment": ["a", "b"] * 250})
current = reference.assign(activity=reference.activity + 100)
numerical = ["activity"]
categorical = ["segment"]
quality_metrics = None  # Example when available: {"rmse": 1.23, "rows_labeled": 500}
context = {"model_id": "usecase2", "environment": "dev", "data_kind": "synthetic_example"}
# For real batches, add model_package_arn, inference_execution_arn, reference_uri and current_uri.
display(current.head())


## Read the local reports

- Compare the unchanged batch and then the deliberately shifted batch with the same reference.
- Write HTML, Evidently JSON, summary and scalar metrics for each comparison.
- Open the report links and inspect which feature changed; a drift flag requests review.
- Leave model quality unavailable until the model owner supplies actual analysis metrics.


In [ ]:
outputs = {}
for case, frame in {"unchanged": reference, "shifted": current}.items():
    folder = ROOT / ".runs/monitoring-example" / case
    summary = write_report(reference, frame, folder, numerical=numerical, categorical=categorical,
                           context={**context, "case": case}, quality_metrics=quality_metrics,
                           threshold=0.2, drift_share=0.5)
    outputs[case] = folder
    print(case, "drift:", summary["drift_detected"], "quality:", summary["quality_status"])
    display(FileLink(os.path.relpath(folder / "report.html", Path.cwd())))


## Optionally publish to existing services

- Set an approved S3 report prefix and enable publication explicitly.
- Optionally create one MLflow analysis run per report in an existing tracking App/server.
- Optionally send a CloudWatch metric and a breach message to an existing approved SNS topic.
- Print the exact report and run references; the unchanged case sends no breach message.

This cell creates no topic, subscription, alarm or scheduler. BMO connects the message to
its response process. Notifications are at least once; receivers can use `summary_uri`
for deduplication. Stop/replay a failed publication after correcting its permissions.


In [ ]:
import boto3
from contextlib import nullcontext
from datetime import datetime, timezone
from mlops_models.pipelines.tracking import managed_client

publish = False
region = "us-east-1"
report_prefix = ""  # An approved s3://bucket/application/monitoring prefix.
tracking_arn = ""  # Optional existing SageMaker MLflow App or tracking-server ARN.
experiment_name = "models/dev/usecase2/monitoring"
cloudwatch_namespace = ""  # Optional approved namespace.
sns_topic_arn = ""  # Optional existing approved topic; breach messages only.

if publish:
    if not report_prefix:
        raise ValueError("Set the report prefix before enabling publication.")
    session = boto3.Session(region_name=region)
    report_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    connection = managed_client(tracking_arn) if tracking_arn else nullcontext(None)
    with connection as client:
        experiment_id = None
        if client is not None:
            experiment = client.get_experiment_by_name(experiment_name)
            experiment_id = experiment.experiment_id if experiment else client.create_experiment(experiment_name)
        for case, folder in outputs.items():
            run_id = None
            if client is not None:
                run_id = client.create_run(experiment_id, run_name=f"monitoring-{case}-{report_id}",
                                           tags={"workflow": "monitoring", **context, "case": case}).info.run_id
            try:
                receipt = publish_report(session, folder, f"{report_prefix.rstrip('/')}/{report_id}/{case}",
                                         client=client, run_id=run_id, cloudwatch_namespace=cloudwatch_namespace,
                                         sns_topic_arn=sns_topic_arn)
            except Exception:
                if client is not None:
                    client.set_terminated(run_id, status="FAILED")
                raise
            else:
                if client is not None:
                    client.set_terminated(run_id, status="FINISHED")
                print(json.dumps(receipt, indent=2))
else:
    print("Local reports only. Enable publication after setting the approved destinations.")
